# CPP features in a scikit-learn ``Pipeline``

AAanalysis features are plain numerical matrices, so they drop straight into a
stock `scikit-learn` workflow — **no AAanalysis-specific estimator or wrapper
class is needed**. The key observation is that
`SequenceFeature().feature_matrix(features, df_parts)` is **stateless**: given a
fixed feature set it deterministically turns sequence parts into an `X` matrix.
A stateless transform is exactly what `sklearn.preprocessing.FunctionTransformer`
wraps, so we can compose CPP features with any sklearn estimator inside a normal
`Pipeline` (see [Breimann25a]_).

This recipe shows the **leakage-aware** pattern:

1. **Discover the CPP feature set once, *outside* cross-validation** — `CPP.run`
   contrasts the two label groups to pick features, so running it inside a CV
   fold would leak that fold's test rows into the selected features.
2. **Put only the deterministic `feature_matrix` transform inside the
   `Pipeline`** — it has no knowledge of the labels and so cannot leak.


## A tiny dataset

We use a small balanced slice of `DOM_GSEC` so the recipe runs in seconds and
needs no `torch` / `fair-esm` / `biopython`. `load_dataset(name=..., n=N)` returns
`N` rows per class (so `2N` total); labels are read from the `'label'` column.

In [1]:
import numpy as np
from functools import partial

from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=15)   # 15 per class -> 30 rows
labels = np.array(df_seq["label"].to_list())

sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
# A small, fixed scale set keeps CPP fast for the demo.
df_scales = aa.load_scales(top60_n=20).T.head(20).T
aa.display_df(df_parts, n_rows=10, show_shape=True)

DataFrame shape: (30, 3)


,tmd,jmd_n_tmd_n,tmd_c_jmd_c
entry,,,
Q14802,LQVGGLICAGVLCAMGIIIVMSA,NSPFYYDWHSLQVGGLICAGVL,CAMGIIIVMSAKCKCKFGQKS
Q86UE4,WVILVGTGALGLLLLFLLGYGWA,LGLEPKRYPGWVILVGTGALGL,LLLFLLGYGWAAACAGARKKR
Q969W9,FVQIIIIVVVMMVMVVVITCLLS,FQSMEITELEFVQIIIIVVVMM,VMVVVITCLLSHYKLSARSFI
P53801,ALIITMSVVGGTLLLGIAICCCC,RWGVCWVNFEALIITMSVVGGT,LLLGIAICCCCCCRRKRSRKP
Q8IUW5,IAYALVPVFFIMGLFGVLICHLL,NDTGNGHPEYIAYALVPVFFIM,GLFGVLICHLLKKKGYRCTTE
P01135,AITALVVVSIVALAVLIITCVLI,AVVAASQKKQAITALVVVSIVA,LAVLIITCVLIHCCQVRKHCE
O43914,VLAGIVMGDLVLTVLIALAVYFL,DCSCSTVSPGVLAGIVMGDLVL,TVLIALAVYFLGRLVPRGRGA
P05556,IIPIVAGVVAGIVLIGLALLLIW,ENPECPTGPDIIPIVAGVVAGI,VLIGLALLLIWKLLMIIHDRR
P16234,VAAAVLVLLVIVIISLIVLVVIW,VAPTLRSELTVAAAVLVLLVIV,IISLIVLVVIWKQKPRYEIRW


## Step 1 — discover the feature set once (outside CV)

`CPP.run` returns the ranked feature table `df_feat`. We compute it **once, on the
full training data, before any cross-validation**. Its `'feature'` column is the
PART-SPLIT-SCALE id set we will hand to every fold.

In [2]:
cpp = aa.CPP(df_parts=df_parts, df_scales=df_scales, verbose=False, random_state=0)
df_feat = cpp.run(labels=labels.tolist(), n_filter=15, n_jobs=1)
aa.display_df(df_feat, n_rows=10, show_shape=True)

/Users/stephanbreimann/Programming/1Packages/aaanalysis/.claude/worktrees/agent-a06f6ec659dacb23b/aaanalysis/feature_engineering/_backend/cpp_run.py:143: UserWarning: CPP is using the Python kernel fallback — the compiled Cython extension is not available in this install. Output is bit-exact with the Cython path but ~2x slower. Reinstall via `pip install --force-reinstall aaanalysis` to fetch a prebuilt wheel.
  warnings.warn(


DataFrame shape: (15, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD-Pattern(C,4,8)-CHOP780212",Conformation,β-sheet (C-term),β-turn (1st residue),"Frequency of th...-Fasman, 1978b)",0.422000,0.225000,-0.225000,0.038000,0.186000,0.000081,0.080527,"23,27"
2,"TMD_C_JMD_C-Pat...4,8)-CHOP780212",Conformation,β-sheet (C-term),β-turn (1st residue),"Frequency of th...-Fasman, 1978b)",0.422000,0.225000,-0.225000,0.038000,0.186000,0.000081,0.040263,"24,28"
3,"TMD_C_JMD_C-Pat...,15)-GEIM800103",Conformation,Unclassified (Conformation),α-helix (β-proteins),"Alpha-helix ind...-Roberts, 1980)",0.407000,0.176000,-0.176000,0.126000,0.067000,0.000147,0.036506,"26,29,33"
4,"TMD_C_JMD_C-Pat...,14)-GEIM800103",Conformation,Unclassified (Conformation),α-helix (β-proteins),"Alpha-helix ind...-Roberts, 1980)",0.407000,0.176000,-0.176000,0.126000,0.067000,0.000147,0.048675,"27,30,34"
5,"TMD_C_JMD_C-Pat...,12)-CHOP780206",Conformation,α-helix (N-cap),Non helical reg...on (N-terminal),"Normalized freq...-Fasman, 1978b)",0.402000,0.171000,-0.171000,0.075000,0.096000,0.000174,0.028743,"24,28,32"
6,"TMD-Segment(10,11)-CHAM820102",Polarity,Hydrophobicity (interface),Free energy (interface),"Free energy of ...-Charton, 1982)",0.400000,0.182000,-0.182000,0.042000,0.127000,0.000189,0.026757,"27,28"
7,"TMD_C_JMD_C-Pat...,14)-CHOP780215",Conformation,β-turn (C-term),β-turn (4th residue),"Frequency of th...-Fasman, 1978b)",0.382000,0.166000,-0.166000,0.082000,0.117000,0.000361,0.023822,"27,30,33,37"
8,"TMD_C_JMD_C-Pat...,14)-CHOP780206",Conformation,α-helix (N-cap),Non helical reg...on (N-terminal),"Normalized freq...-Fasman, 1978b)",0.364000,0.176000,-0.176000,0.109000,0.116000,0.000671,0.030191,"27,31"
9,"TMD_C_JMD_C-Pat...,12)-CHOP780206",Conformation,α-helix (N-cap),Non helical reg...on (N-terminal),"Normalized freq...-Fasman, 1978b)",0.356000,0.148000,-0.148000,0.078000,0.128000,0.000906,0.034492,"25,29,32"
10,"TMD-Pattern(C,4...,15)-FASG760105",Polarity,Unclassified (Polarity),pK-C,"pK-C (Fasman, 1976)",0.353000,0.141000,0.141000,0.048000,0.109000,0.000975,0.035767,"16,20,24,27"


## Step 2 — wrap the stateless transform in a ``FunctionTransformer``

`feature_matrix` only needs the (already discovered) feature ids and the same
scales; the per-fold input is just `df_parts`. We bind the fixed arguments with
`functools.partial` and wrap the result in a stock `FunctionTransformer` — this is
an ordinary sklearn transformer with `fit` / `transform`, **no new AAanalysis
symbol involved**.

In [3]:
def cpp_feature_matrix(df_parts, *, features, df_scales):
    """Stateless: turn sequence parts into the CPP feature matrix X."""
    return aa.SequenceFeature().feature_matrix(
        features=features, df_parts=df_parts, df_scales=df_scales, n_jobs=1)

cpp_transformer = FunctionTransformer(
    func=partial(cpp_feature_matrix, features=df_feat["feature"], df_scales=df_scales))

pipe = Pipeline([
    ("cpp_feats", cpp_transformer),
    ("clf", RandomForestClassifier(n_estimators=50, random_state=0)),
])
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cpp_feats', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",functools.par... 0.857 )
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments to pass to func... 

## Step 3 — fit and predict

The `Pipeline` is handed `df_parts` (the per-sample sequence parts) as `X`; the
`cpp_feats` step turns it into the numeric feature matrix and feeds the classifier.

In [4]:
pipe.fit(df_parts, labels)
y_pred = pipe.predict(df_parts)
print(f"n predictions: {len(y_pred)}")
print(f"training-set accuracy: {pipe.score(df_parts, labels):.3f}")

n predictions: 30
training-set accuracy: 1.000


## The transform is byte-identical to a direct call

Routing the parts through the `Pipeline` introduces **no drift**: the `X` the
transformer produces is byte-for-byte identical to calling
`SequenceFeature().feature_matrix(...)` directly. There is one column per
discovered feature.

In [5]:
X_pipe = pipe.named_steps["cpp_feats"].transform(df_parts)
X_direct = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts,
                             df_scales=df_scales, n_jobs=1)
print(f"X shape: {X_pipe.shape}  (n_features == len(df_feat): "
      f"{X_pipe.shape[1] == len(df_feat)})")
print(f"byte-identical to direct feature_matrix call: "
      f"{np.asarray(X_pipe).tobytes() == np.asarray(X_direct).tobytes()}")

X shape: (30, 15)  (n_features == len(df_feat): True)
byte-identical to direct feature_matrix call: True


## Cross-validate the leakage-aware way

Because feature discovery already happened **once, above**, we can safely run the
`Pipeline` inside `cross_val_score`: only the deterministic, label-blind
`feature_matrix` transform runs per fold. Each of the 5 folds yields one score.

In [6]:
import pandas as pd

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(pipe, df_parts, labels, cv=cv, scoring="accuracy")
df_scores = pd.DataFrame({"fold": range(1, len(scores) + 1),
                          "accuracy": scores.round(3)})
aa.display_df(df_scores, n_rows=10, show_shape=True)
print(f"mean CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

DataFrame shape: (5, 2)


,fold,accuracy
1,1,0.667000
2,2,1.000000
3,3,0.833000
4,4,1.000000
5,5,1.000000


mean CV accuracy: 0.900 +/- 0.133


## Takeaways

- CPP features compose into a **stock** `sklearn.pipeline.Pipeline` via
  `FunctionTransformer` — no AAanalysis-specific wrapper class.
- The `feature_matrix` transform is **stateless and byte-stable**, so it is safe
  inside a `Pipeline` and inside `cross_val_score`.
- **Leakage-aware rule of thumb:** run `CPP.run` feature *discovery* **once,
  outside** cross-validation; keep only the deterministic `feature_matrix`
  transform **inside** the `Pipeline`.